In [ ]:
!pip install mlflow

In [ ]:
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold,GroupShuffleSplit
from collections import Counter, defaultdict
import pandas as pd
from tqdm.auto import tqdm
import mlflow
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
TRACKING_URI = " https://average-candied-vendor.ngrok-free.dev"
EXPERIMENT = "rubert_finetuning"
RUN_NAME = "ruslit df contrasive loss rubert-base different params"

MLFLOW_PREPROCESS_RUN_ID = "406f39f4f7844e10b58b1b508d6f2bc7"
#MLFLOW_PREPROCESS_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
#MLFLOW_DF_ARTIFACT_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
#MLFLOW_DF_ARTIFACT_PATH = "data/multigenres_df.csv"

RANDOM_STATE = 42

In [ ]:
# Детерминизм для воспроизводимости
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

torch.backends.cudnn.benchmark = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"
#MODEL_NAME = "cointegrated/rubert-tiny2"  # быстрый вариант для экспериментов
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
from mlflow.tracking import MlflowClient

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.start_run(run_name=RUN_NAME)

mlflow.set_tag("upstream.preprocess_run_id", MLFLOW_PREPROCESS_RUN_ID)
mlflow.log_param("random_state", RANDOM_STATE)
mlflow.log_param("device", DEVICE)

client = MlflowClient()

In [ ]:
#local_path = client.download_artifacts(run_id=MLFLOW_DF_ARTIFACT_RUN_ID, path=MLFLOW_DF_ARTIFACT_PATH)
local_path = "balanced_lit.csv"
data = pd.read_csv(local_path)
#data = data[data['source_type'] == 'socials']
data.head()

In [ ]:
data.author.unique()

In [ ]:
#EXCLUDED_AUTHORS_LIST = ['Bulgakov', 'Роулинг', 'Turgenev', '500600640', 'Соня Долотовская']
EXCLUDED_AUTHORS_LIST = ['Bulgakov', 'Роулинг', 'Turgenev']
data = data[~data.author.isin(EXCLUDED_AUTHORS_LIST)]
mlflow.log_param('excluded_authors1', ','.join(EXCLUDED_AUTHORS_LIST))

In [ ]:
# чтобы не считать эмбеддинги заново каждый раз + поиск
class EmbeddingStore:

    def __init__(self):
        self.embeddings: list = []
        self.authors:    list = []
        self.texts:      list = []
        self.sources:    list = []
        self._matrix_cache = None
        self._dirty: bool = True

    def add(self, author: str, embedding: np.ndarray,
            text: str = '', source: str = 'unknown'):
        self.embeddings.append(embedding)
        self.authors.append(author)
        self.texts.append(text)
        self.sources.append(source)
        self._dirty = True

    def get_matrix(self) -> np.ndarray:
        if self._dirty or self._matrix_cache is None:
            self._matrix_cache = np.stack(self.embeddings)
            self._dirty = False
        return self._matrix_cache

    def get_authors(self) -> list:
        return self.authors

    def search(self, query: np.ndarray, k: int = 5) -> list:
        # k-nearest-neighbours по косинусному сходству.
        if not self.embeddings:
            return []
        matrix = self.get_matrix()
        sims   = matrix @ query.reshape(-1, 1)
        sims   = sims.squeeze()
        top_k  = np.argsort(sims)[::-1][:k]
        return [
            {'author':     self.authors[i],
             'similarity': float(sims[i]),
             'text':       self.texts[i][:80]}
            for i in top_k
        ]

    def __len__(self):
        return len(self.embeddings)

In [ ]:
class SiameseBERT(nn.Module):

    def __init__(self, model_name: str = 'DeepPavlov/rubert-base-cased',
                 dropout: float = 0.2,
                 hidden_head_size: int = 512,
                 embedding_dim: int = 256):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_head_size),
            nn.LayerNorm(hidden_head_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_head_size, embedding_dim)
        )

    def encode(self, input_ids: torch.Tensor,
           attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = out.last_hidden_state

        mask_expanded = attention_mask.unsqueeze(-1).float()

        sum_embeddings = (token_embeddings * mask_expanded).sum(dim=1)

        token_counts = mask_expanded.sum(dim=1).clamp(min=1e-9)

        pooled = sum_embeddings / token_counts

        projected = self.projection(pooled)

        _result = F.normalize(projected, p=2, dim=-1)

        return _result

    def forward(self, input_ids, attention_mask):
        return self.encode(input_ids, attention_mask)

In [ ]:
def freeze_lower_layers(model: SiameseBERT, freeze_until_layer: int = 8):

    for param in model.bert.embeddings.parameters():
        param.requires_grad = False

    for i, layer in enumerate(model.bert.encoder.layer):
        if i < freeze_until_layer:
            for param in layer.parameters():
                param.requires_grad = False

In [ ]:
# вместо троек в батче считаем сразу несколько текстов от n авторов
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        # разделение тоже будет обучаемо
        self.log_temperature = nn.Parameter(torch.tensor(temperature).log())

    @property
    def temperature(self):
        return self.log_temperature.exp().clamp(min=0.01, max=2.0)

    def forward(self, embeddings, labels):
        device     = embeddings.device
        batch_size = embeddings.shape[0]

        sim_matrix = torch.matmul(embeddings, embeddings.T) / self.temperature

        # не считаем текст сам с собой
        labels_col  = labels.unsqueeze(1)
        positive_mask = (labels_col == labels_col.T).float()
        positive_mask.fill_diagonal_(0)

        # все пары кроме диагонали(сам с собой)
        identity = torch.eye(batch_size, device=device)
        denominator_mask = 1 - identity

        logits_max, _ = sim_matrix.max(dim=1, keepdim=True)
        logits  = sim_matrix - logits_max.detach()

        exp_logits = torch.exp(logits) * denominator_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-9)

        n_positives = positive_mask.sum(dim=1)
        loss = -(positive_mask * log_prob).sum(dim=1) / (n_positives + 1e-9)

        return loss.mean()

In [ ]:
class SupConDataset(Dataset):
    def __init__(self, corpus, source_corpus, tokenizer,
                 n_authors=16, k_shots=4,
                 max_len=256, size=3000,
                 same_source=True):


        self.author_chunks = {a: c for a, c in corpus.items() if len(c) >= k_shots}
        self.author_source = {a: source_corpus.get(a, 'unknown')
                              for a in self.author_chunks}
        self.authors = list(self.author_chunks.keys())

        self.source_to_authors = defaultdict(list)
        for a, src in self.author_source.items():
            self.source_to_authors[src].append(a)

        print("Предтокенизация корпуса...")
        all_texts = []
        for chunks in self.author_chunks.values():
            all_texts.extend(chunks)

        BATCH = 512
        ids_cache, masks_cache = [], []
        for i in tqdm(range(0, len(all_texts), BATCH), desc="Токенизация"):
            enc = tokenizer(
                all_texts[i:i+BATCH],
                max_length=max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            ids_cache.append(enc['input_ids'])
            masks_cache.append(enc['attention_mask'])

        all_ids = torch.cat(ids_cache, dim=0)
        all_masks = torch.cat(masks_cache, dim=0)

        # Строим кеш по тексту для быстрого доступа по метке
        # text_to_idx[текст] = индекс в all_ids/all_masks
        text_to_idx = {}
        offset = 0
        self.token_cache = {}
        for author, chunks in self.author_chunks.items():
            n = len(chunks)
            self.token_cache[author] = {
                'ids':    all_ids[offset : offset + n],
                'masks':  all_masks[offset : offset + n],
                'chunks': chunks
            }
            for j, chunk in enumerate(chunks):
                text_to_idx[chunk] = offset + j
            offset += n

        self.n_authors  = n_authors
        self.k_shots = k_shots
        self.size = size
        self.same_source = same_source

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if self.same_source:
            return self._get_same_source_batch()
        else:
            return self._get_random_batch()

    def _get_same_source_batch(self):
        # чтобы модель брала тексты из одного жанра, а не из разных
        src    = random.choice(list(self.source_to_authors.keys()))
        pool   = self.source_to_authors[src]
        chosen = random.sample(pool, min(self.n_authors, len(pool)))
        return self._build_batch(chosen)

    def _get_random_batch(self):
        chosen = random.sample(self.authors, min(self.n_authors, len(self.authors)))
        return self._build_batch(chosen)

    def _build_batch(self, chosen):
        ids_list, masks_list, labels_list = [], [], []
        for label_idx, author in enumerate(chosen):
            cache   = self.token_cache[author]
            n_avail = cache['ids'].shape[0]
            idxs    = random.sample(range(n_avail), min(self.k_shots, n_avail))
            ids_list.append(cache['ids'][idxs])
            masks_list.append(cache['masks'][idxs])
            labels_list.extend([label_idx] * len(idxs))
        return (torch.cat(ids_list,   dim=0),
                torch.cat(masks_list, dim=0),
                torch.tensor(labels_list))

def supcon_collate(batch):
    """берём только первый из data loader"""
    ids, masks, labels = batch[0]
    return ids, masks, labels

In [ ]:
def df_to_corpus(df: pd.DataFrame,
                 text_col:   str = 'text',
                 author_col: str = 'author') -> dict:
    return {
        author: group[text_col].tolist()
        for author, group in df.groupby(author_col)
    }


def df_to_source_corpus(df: pd.DataFrame,
                        author_col: str = 'author',
                        source_col: str = 'source_type') -> dict:
    return {
        author: group[source_col].tolist()[0]
        for author, group in df.groupby(author_col)
    }

def df_to_source_corpus_man(df: pd.DataFrame,
                        author_col: str = 'author',
                        source_name: str = 'ruslit') -> dict:
    # для датасетов без source_type
    return {
        author: source_name for author, group in df.groupby(author_col)
    }

def split_corpus(df: pd.DataFrame,
                 val_size: float = 0.2) -> tuple:
    # фрагменты одного исходного текста  попадают только в train или только в val.

    groups = df['source_id'] if 'source_id' in df.columns else df['author']
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=RANDOM_STATE)
    train_idx, val_idx = next(splitter.split(df, groups=groups))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    print(f"Train: {len(train_df)} фрагментов, {train_df['author'].nunique()} авторов")
    print(f"Val:   {len(val_df)} фрагментов, {val_df['author'].nunique()} авторов")
    return train_df, val_df

In [ ]:
def build_embedding_store(model: SiameseBERT,
                           corpus: dict,
                           source_corpus: dict,
                           tokenizer,
                           device: str = 'cuda',
                           batch_size: int = 64,
                           max_len: int = 256) -> EmbeddingStore:
    store = EmbeddingStore()
    model.eval()

    all_texts, all_authors, all_sources = [], [], []
    for author, chunks in corpus.items():
        src = source_corpus.get(author, 'unknown')
        for chunk in chunks:
            all_texts.append(chunk)
            all_authors.append(author)
            all_sources.append(src)

    with torch.no_grad():
        for i in range(0, len(all_texts), batch_size):
            batch_texts = all_texts[i : i + batch_size]
            enc = tokenizer(
                batch_texts, max_length=max_len,
                padding=True, truncation=True,
                return_tensors='pt'
            )
            with autocast():
                embs = model.encode(
                    enc['input_ids'].to(device),
                    enc['attention_mask'].to(device)
                ).float().cpu().numpy()

            for j, emb in enumerate(embs):
                store.add(
                    author=all_authors[i + j],
                    embedding=emb,
                    text=batch_texts[j],
                    source=all_sources[i + j]
                )

    print(f"Store: {len(store)} эмбеддингов, {len(corpus)} авторов")
    return store


def build_weighted_centroids(store: EmbeddingStore) -> EmbeddingStore:
    # центроид автора = взвешенное среднее его эмбеддингов
    author_data = defaultdict(list)

    for i, author in enumerate(store.authors):
        weight = min(len(store.texts[i]), 2000) / 2000.0
        author_data[author].append((store.embeddings[i], weight, store.sources[i]))

    centroid_store = EmbeddingStore()

    for author, items in author_data.items():
        embs, weights, sources = zip(*items)
        weights  = np.array(weights)
        weights  = weights / weights.sum()
        matrix   = np.stack(embs)
        centroid = (matrix * weights[:, np.newaxis]).sum(axis=0)
        norm     = np.linalg.norm(centroid)
        centroid = centroid / norm if norm > 0 else centroid

        centroid_store.add(
            author    = author,
            embedding = centroid,
            text      = f'[centroid: {len(items)} фрагментов]',
            source    = sources[0]
        )

    print(f"Centroid store: {len(centroid_store)} авторов")
    return centroid_store

In [ ]:
def compute_full_retrieval_metrics(model: SiameseBERT,
                                   corpus: dict,
                                   store: EmbeddingStore,
                                   tokenizer,
                                   k_values: list = [1, 3, 5, 10],
                                   device: str = 'cuda',
                                   batch_size: int = 64,
                                   max_len: int = 256) -> dict:
    """
    Вычисляет полный набор метрик для retrieval + 1-NN классификации.

    Returns dict с ключами:
        map@k, mrr, precision@k, recall@k  для каждого k в k_values
        f1_macro, f1_weighted, precision_macro, recall_macro (1-NN классификация)
        per_author_f1: dict автор -> f1
    """
    model.eval()

    all_chunks, all_true_authors = [], []
    for author, chunks in corpus.items():
        for chunk in chunks:
            all_chunks.append(chunk)
            all_true_authors.append(author)

    # Батчевый inference запросов
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(all_chunks), batch_size):
            batch = all_chunks[i : i + batch_size]
            enc   = tokenizer(batch, max_length=max_len,
                              padding=True, truncation=True,
                              return_tensors='pt')
            with autocast():
                emb = model.encode(enc['input_ids'].to(device),
                                   enc['attention_mask'].to(device)).float()
            all_embeddings.append(emb.cpu().numpy())

    query_matrix  = np.vstack(all_embeddings)      # (N, dim)
    store_matrix  = store.get_matrix()             # (M, dim)
    store_authors = store.get_authors()

    sim_matrix = query_matrix @ store_matrix.T     # (N, M) — все косинусные сходства сразу

    max_k = max(k_values)

    # Аккумуляторы
    ap_by_k      = {k: [] for k in k_values}
    prec_by_k    = {k: [] for k in k_values}
    recall_by_k  = {k: [] for k in k_values}
    rr_scores    = []    # reciprocal rank
    pred_authors = []    # предсказание 1-NN

    for i in range(len(all_chunks)):
        true_author = all_true_authors[i]
        chunk       = all_chunks[i]

        # Сортируем по убыванию сходства, исключаем сам текст (exact match)
        sorted_idx = np.argsort(sim_matrix[i])[::-1]
        # Убираем exact match: сравниваем начало текста для скорости
        sorted_idx = [j for j in sorted_idx
                      if store.texts[j][:60] != chunk[:60]]

        # 1-NN классификация: ближайший сосед (после исключения самого себя)
        if sorted_idx:
            pred_authors.append(store_authors[sorted_idx[0]])
        else:
            pred_authors.append('unknown')

        # MRR: ищем первый правильный ответ
        rr = 0.0
        for rank, j in enumerate(sorted_idx[:max_k + 1]):
            if store_authors[j] == true_author:
                rr = 1.0 / (rank + 1)
                break
        rr_scores.append(rr)

        # MAP@k, Precision@k, Recall@k для каждого k
        # Считаем сколько вообще правильных в store (для recall)
        total_relevant = sum(1 for a in store_authors if a == true_author)

        for k in k_values:
            top_k = sorted_idx[:k]
            correct, precisions = 0, []

            for rank, j in enumerate(top_k):
                if store_authors[j] == true_author:
                    correct += 1
                    precisions.append(correct / (rank + 1))

            ap_by_k[k].append(np.mean(precisions) if precisions else 0.0)
            prec_by_k[k].append(correct / k)
            recall_by_k[k].append(correct / max(total_relevant, 1))

    # ── Метрики классификации через 1-NN ──────────────────────────────────────
    # Отбираем только авторов, которые есть и в запросах, и в store
    common_authors = sorted(set(all_true_authors) & set(store_authors))
    # Для classification_report нужно чтобы все labels были известны
    report = classification_report(
        all_true_authors, pred_authors,
        labels=common_authors,
        output_dict=True,
        zero_division=0
    )

    per_author_f1 = {a: report[a]['f1-score'] for a in common_authors if a in report}

    results = {
        'mrr':              float(np.mean(rr_scores)),
        'f1_macro':         report['macro avg']['f1-score'],
        'f1_weighted':      report['weighted avg']['f1-score'],
        'precision_macro':  report['macro avg']['precision'],
        'recall_macro':     report['macro avg']['recall'],
        'per_author_f1':    per_author_f1,
    }
    for k in k_values:
        results[f'map@{k}']       = float(np.mean(ap_by_k[k]))
        results[f'precision@{k}'] = float(np.mean(prec_by_k[k]))
        results[f'recall@{k}']    = float(np.mean(recall_by_k[k]))

    return results


def print_metrics(metrics: dict, title: str = 'Метрики', k_values: list = [1, 3, 5]):
    """Красивый вывод словаря метрик."""
    print(f"\n{'─'*50}")
    print(f"  {title}")
    print(f"{'─'*50}")
    print(f"  MRR:               {metrics['mrr']:.4f}")
    print(f"  F1 macro:          {metrics['f1_macro']:.4f}")
    print(f"  F1 weighted:       {metrics['f1_weighted']:.4f}")
    print(f"  Precision macro:   {metrics['precision_macro']:.4f}")
    print(f"  Recall macro:      {metrics['recall_macro']:.4f}")
    print()
    for k in k_values:
        if f'map@{k}' in metrics:
            print(f"  MAP@{k}:            {metrics[f'map@{k}']:.4f}  "
                  f"P@{k}: {metrics[f'precision@{k}']:.4f}  "
                  f"R@{k}: {metrics[f'recall@{k}']:.4f}")
    print(f"{'─'*50}")

def log_metrics_mlflow(metrics: dict, title: str = 'Метрики', k_values: list = [1, 3, 5]):
    """Красивый вывод словаря метрик."""
    mlflow.log_metric(f"MRR_{title}", metrics['mrr'])
    mlflow.log_metric(f"F1_macro_{title}", metrics['f1_macro'])
    mlflow.log_metric(f"F1_weighted_{title}", metrics['f1_weighted'])
    mlflow.log_metric(f"precision_macro_{title}", metrics['precision_macro'])
    mlflow.log_metric(f"recall_macro_{title}", metrics['recall_macro'])

    for k in k_values:
        if f'map@{k}' in metrics:
            mlflow.log_metric(f"map_at_{k}_{title}", metrics[f'map@{k}'])
            mlflow.log_metric(f"precision_at_{k}_{title}", metrics[f'precision@{k}'])
            mlflow.log_metric(f"recall_at_{k}_{title}", metrics[f'recall@{k}'])


def compare_fragment_vs_centroid_metrics(model, corpus, fragment_store,
                                          centroid_store, tokenizer,
                                          k_values=(1, 3, 5),
                                          device='cuda'):
    """
    Сравнивает качество retrieval по фрагментам и по центроидам.

    При поиске по центроидам каждый автор представлен одним вектором —
    это быстрее, но теряет информацию об отдельных текстах.
    Сравнение помогает понять, стоит ли усреднять эмбеддинги.
    """
    print("Вычисление метрик по фрагментам...")
    frag_metrics = compute_full_retrieval_metrics(
        model, corpus, fragment_store, tokenizer,
        k_values=list(k_values), device=device
    )
    print("Вычисление метрик по центроидам...")
    cent_metrics = compute_full_retrieval_metrics(
        model, corpus, centroid_store, tokenizer,
        k_values=list(k_values), device=device
    )

    print_metrics(frag_metrics, title='Fragment Store', k_values=list(k_values))
    print_metrics(cent_metrics, title='Centroid Store', k_values=list(k_values))

    log_metrics_mlflow(cent_metrics, title='framgents', k_values=list(k_values))
    log_metrics_mlflow(cent_metrics, title='centroids', k_values=list(k_values))

    # Дельта: положительная = centroid лучше
    print("\n  Δ (centroid − fragment):")
    for key in ['mrr', 'f1_macro'] + [f'map@{k}' for k in k_values]:
        delta = cent_metrics[key] - frag_metrics[key]
        sign  = '+' if delta >= 0 else ''
        print(f"    {key:15s}: {sign}{delta:.4f}")

    return frag_metrics, cent_metrics


def evaluate_by_source(model, corpus_df, fragment_store, centroid_store,
                       tokenizer, k: int = 5, device: str = 'cuda'):
    """MAP@k и F1 отдельно по каждому источнику."""
    print(f"\nМетрики по источникам (MAP@{k} / F1-macro):")
    for src in corpus_df['source_type'].unique():
        src_df     = corpus_df[corpus_df['source_type'] == src]
        src_corpus = df_to_corpus(src_df)

        frag_m = compute_full_retrieval_metrics(
            model, src_corpus, fragment_store, tokenizer,
            k_values=[k], device=device
        )
        cent_m = compute_full_retrieval_metrics(
            model, src_corpus, centroid_store, tokenizer,
            k_values=[k], device=device
        )
        print(f"  {src:10s}: MAP@{k}={frag_m[f'map@{k}']:.4f} (frag) "
              f"| {cent_m[f'map@{k}']:.4f} (centroid) "
              f"| F1={frag_m['f1_macro']:.4f}  "
              f"({src_df['author'].nunique()} авторов)")

In [ ]:
def train(model: SiameseBERT,
          train_loader: DataLoader,
          val_loader: DataLoader,
          val_corpus: dict,
          val_source_corpus: dict,
          tokenizer,
          epochs: int        = 15,
          bert_lr: float     = 2e-5,
          head_lr: float     = 1e-3,
          temperature: float = 0.07,
          device: str        = 'cuda',
          patience: int      = 4,
          eval_map_every: int = 1,
          log_every: int      = 50,
          use_amp: bool       = True,
          grad_accum: int     = 1,
          k_values: list      = [1, 3, 5]
          ) -> tuple:
    model = model.to(device)

    criterion = SupervisedContrastiveLoss(temperature=temperature).to(device)

    optimizer = torch.optim.AdamW([
        {'params': model.bert.parameters(),        'lr': bert_lr,  'weight_decay': 0.01},
        {'params': model.projection.parameters(),  'lr': head_lr,  'weight_decay': 0.005},
        {'params': criterion.parameters(),         'lr': head_lr,  'weight_decay': 0.0},
    ])

    mlflow.log_params({'weight_decay_bert': 0.01, 'weight_decay_head': 0.005})

    effective_steps = (len(train_loader) // grad_accum) * epochs
    warmup_steps    = effective_steps // 10

    mlflow.log_params({'effective_steps': effective_steps, 'warmup_steps': warmup_steps})

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=effective_steps
    )

    scaler = GradScaler(enabled=use_amp)

    best_map5 = 0.0
    patience_counter = 0
    history = []

    total_steps = len(train_loader)
    print(f"\nНачало обучения: {epochs} эпох × {total_steps} шагов/эпоха")
    print(f"AMP: {'включён (fp16)' if use_amp else 'выключен (fp32)'}")
    print(f"Gradient accumulation: {grad_accum}")
    print(f"Effective batch size: {train_loader.dataset.n_authors * train_loader.dataset.k_shots * grad_accum}")
    print(f"Warmup: {warmup_steps} шагов, Total: {effective_steps} шагов")
    print(f"Early stopping: MAP@5, patience={patience}\n")

    for epoch in range(1, epochs + 1):

        model.train()
        criterion.train()
        train_losses = []
        epoch_start  = time.time()
        step_start   = time.time()

        pbar = tqdm(enumerate(train_loader), total=total_steps,
                    desc=f"Epoch {epoch:02d}/{epochs} [train]",
                    leave=False)

        optimizer.zero_grad()
        for step, (ids, masks, labels) in pbar:
            ids    = ids.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(enabled=use_amp):
                emb  = model.encode(ids, masks)
                loss = criterion(emb, labels)
                loss = loss / grad_accum

            scaler.scale(loss).backward()

            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) + list(criterion.parameters()),
                    max_norm=1.0
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            train_losses.append(loss.item() * grad_accum)

            if (step + 1) % log_every == 0:
                elapsed = time.time() - step_start
                steps_per_s = log_every / elapsed
                remaining_s = (total_steps - step - 1) / steps_per_s
                current_lr_bert = optimizer.param_groups[0]['lr']
                current_lr_head = optimizer.param_groups[1]['lr']
                recent_loss = np.mean(train_losses[-log_every:])

                if DEVICE == 'cuda':
                    mem_used = torch.cuda.memory_allocated() / 1e9
                    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
                    mem_str = f" | GPU {mem_used:.1f}/{mem_total:.1f}GB"
                else:
                    mem_str = ""

                tqdm.write(
                    f"  Ep {epoch:02d} step {step+1:4d}/{total_steps} "
                    f"| loss={recent_loss:.4f} "
                    f"| τ={criterion.temperature.item():.4f} "
                    f"| lr_bert={current_lr_bert:.1e} lr_head={current_lr_head:.1e} "
                    f"| {steps_per_s:.1f} it/s "
                    f"| ETA {remaining_s/60:.1f} min"
                    f"{mem_str}"
                )
                step_start = time.time()

            pbar.set_postfix({'loss': f'{np.mean(train_losses):.4f}', 'τ': f'{criterion.temperature.item():.3f}'})

        pbar.close()

        model.eval()
        criterion.eval()
        val_losses = []

        torch.manual_seed(RANDOM_STATE)
        np.random.seed(RANDOM_STATE)
        with torch.no_grad():
            for ids, masks, labels in tqdm(val_loader,
                                           desc=f"Epoch {epoch:02d}/{epochs} [val  ]",
                                           leave=False):
                ids = ids.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with autocast(enabled=use_amp):
                    emb  = model.encode(ids, masks)
                    loss = criterion(emb, labels)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)
        epoch_time = time.time() - epoch_start
        current_temp = criterion.temperature.item()

        mlflow.log_metric('epoch_time',  epoch_time,   step=epoch)
        mlflow.log_metric('train_loss',  train_loss,   step=epoch)
        mlflow.log_metric('val_loss',    val_loss,     step=epoch)
        mlflow.log_metric('temperature', current_temp, step=epoch)
        mlflow.log_metric('current_lr_bert', optimizer.param_groups[0]['lr'], step=epoch)
        mlflow.log_metric('current_lr_head', optimizer.param_groups[1]['lr'], step=epoch)

        entry = {
            'epoch':       epoch,
            'train_loss':  train_loss,
            'val_loss':    val_loss,
            'temperature': current_temp,
            'epoch_time':  epoch_time,
        }

        if epoch % eval_map_every == 0 or epoch == epochs:
            current_store  = build_embedding_store(
                model, val_corpus, val_source_corpus, tokenizer, device=device
            )
            centroid_store = build_weighted_centroids(current_store)

            frag_metrics = compute_full_retrieval_metrics(
                model, val_corpus, current_store, tokenizer,
                k_values=k_values, device=device
            )
            cent_metrics = compute_full_retrieval_metrics(
                model, val_corpus, centroid_store, tokenizer,
                k_values=k_values, device=device
            )

            entry['frag_metrics'] = frag_metrics
            entry['cent_metrics'] = cent_metrics

            k_main   = k_values[1] if len(k_values) > 1 else k_values[0]
            map5_val = frag_metrics['map@5']

            print(f"\nEpoch {epoch:02d}/{epochs} "
                  f"| train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  f"| τ={current_temp:.4f} | {epoch_time:.0f}s")
            print(f"  Fragment:  MAP@{k_main}={frag_metrics[f'map@{k_main}']:.4f}  "
                  f"MAP@5={map5_val:.4f}  "
                  f"MRR={frag_metrics['mrr']:.4f}  "
                  f"F1={frag_metrics['f1_macro']:.4f}")
            print(f"  Centroid:  MAP@{k_main}={cent_metrics[f'map@{k_main}']:.4f}  "
                  f"MAP@5={cent_metrics['map@5']:.4f}  "
                  f"MRR={cent_metrics['mrr']:.4f}  "
                  f"F1={cent_metrics['f1_macro']:.4f}")

            mlflow.log_metric(f'P_at_{k_main}', frag_metrics[f'precision@{k_main}'], step=epoch)

            if map5_val > best_map5:
                best_map5        = map5_val
                patience_counter = 0
                torch.save(model.state_dict(), 'best_model.pt')
                print(f"  ✓ Сохранено (MAP@5: {map5_val:.4f}, τ={current_temp:.4f})")
            else:
                patience_counter += 1
                print(f"  ✗ MAP@5 не улучшился {patience_counter}/{patience}")
                if patience_counter >= patience:
                    print(f"\nEarly stopping на эпохе {epoch} (лучший MAP@5: {best_map5:.4f})")
                    break
        else:
            print(f"Epoch {epoch:02d}/{epochs} "
                  f"| train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  f"| τ={current_temp:.4f} | {epoch_time:.0f}s")

        history.append(entry)

    mlflow.log_metric('best_map5', best_map5)

    model.load_state_dict(torch.load('best_model.pt', map_location=device))
    print(f"\nЛучшая модель загружена (MAP@5: {best_map5:.4f})")
    return model, history

In [ ]:
def get_embedding(text: str, model: SiameseBERT,
                  tokenizer, device: str = 'cuda',
                  max_len: int = 256) -> np.ndarray:
    model.eval()
    enc = tokenizer(text, max_length=max_len,
                    padding='max_length', truncation=True, return_tensors='pt')
    with torch.no_grad():
        with autocast():
            emb = model.encode(enc['input_ids'].to(device),
                               enc['attention_mask'].to(device)).float()
    return emb.cpu().numpy().squeeze()


def predict(text: str,
            model: SiameseBERT,
            fragment_store: EmbeddingStore,
            centroid_store: EmbeddingStore,
            tokenizer,
            device: str            = 'cuda',
            k: int                 = 5,
            threshold: float       = 0.3,
            fragment_weight: float = 0.35,
            centroid_weight: float = 0.65) -> dict:
    emb = get_embedding(text, model, tokenizer, device)

    frag_results = fragment_store.search(emb, k=k * 5)
    frag_scores  = defaultdict(float)
    for r in frag_results:
        frag_scores[r['author']] = max(frag_scores[r['author']], r['similarity'])

    centroid_results = centroid_store.search(emb, k=k)
    centroid_scores  = {r['author']: r['similarity'] for r in centroid_results}

    all_auth     = set(frag_scores) | set(centroid_scores)
    final_scores = {
        a: fragment_weight * frag_scores.get(a, 0) +
           centroid_weight * centroid_scores.get(a, 0)
        for a in all_auth
    }

    ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    best_author, best_score = ranked[0]
    predicted = best_author if best_score >= threshold else 'unknown'

    return {
        'predicted':  predicted,
        'confidence': round(best_score, 4),
        'top_k':      [(a, round(s, 4)) for a, s in ranked[:k]],
        'unknown':    predicted == 'unknown',
        'centroid_scores': centroid_scores,
        'frag_scores': frag_scores
    }

In [ ]:
USE_AMP       = True # mixed precision
FREEZE_LAYERS = 8

N_AUTHORS     = 20 # авторов в батче
K_SHOTS       = 5 # фрагментов на автора

MAX_LEN       = 512
TRAIN_SIZE    = 3000 # батчей за эпоху
VAL_SIZE      = 500

EPOCHS        = 10
K_VALUES      = [1, 3, 5]
TEMPERATURE = 0.3

HIDDEN_LAYER_SIZE = 512
DROPOUT = 0.2
EMBEDDIN_DIM = 256

PATIENCE = 6

mlflow.log_params(
    {
        'freeze_layers': FREEZE_LAYERS,
        'authors_in_batch': N_AUTHORS,
        'k_shots_batch': K_SHOTS,
        'train_size_batches': TRAIN_SIZE,
        'val_size_batches': VAL_SIZE,
        'epochs': VAL_SIZE,
        'temperature': TEMPERATURE,
        'max_token_text_len': MAX_LEN,
        'tokenzier': MODEL_NAME,
        'device': DEVICE,
        'hidden_layer_size': HIDDEN_LAYER_SIZE,
        'dropout': DROPOUT,
        'embedding_dim': EMBEDDIN_DIM,
        'patience_before_early_stop': PATIENCE
    }
)
mlflow_log_dataset = mlflow.data.from_pandas(data, name="ruslit balanced")
mlflow.log_input(mlflow_log_dataset)

In [ ]:
source_corpus = df_to_source_corpus_man(data, 'author')
source_corpus

In [ ]:
train_df, val_df = split_corpus(data)

train_corpus = df_to_corpus(train_df)
val_corpus   = df_to_corpus(val_df)

model = SiameseBERT(model_name=MODEL_NAME, embedding_dim=EMBEDDIN_DIM, dropout=DROPOUT, hidden_head_size=HIDDEN_LAYER_SIZE)
freeze_lower_layers(model, freeze_until_layer=FREEZE_LAYERS)

train_ds = SupConDataset(
    train_corpus, source_corpus, tokenizer,
    n_authors=N_AUTHORS, k_shots=K_SHOTS,
    max_len=MAX_LEN, size=TRAIN_SIZE,
     same_label=False,
)
val_ds = SupConDataset(
    val_corpus, source_corpus, tokenizer,
    n_authors=N_AUTHORS, k_shots=K_SHOTS,
    max_len=MAX_LEN, size=VAL_SIZE,
     same_label=False,
)


train_loader = DataLoader(train_ds, batch_size=1,
                          collate_fn=supcon_collate,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=1,
                          collate_fn=supcon_collate,
                          num_workers=2, pin_memory=True)


start = time.perf_counter()
model, history = train(
    model = model,
    train_loader = train_loader,
    val_loader = val_loader,
    val_corpus = val_corpus,
    val_source_corpus = source_corpus,
    tokenizer  = tokenizer,
    epochs = EPOCHS,
    device = DEVICE,
    k_values = K_VALUES,
    use_amp  = USE_AMP,
    log_every = 1000,
    temperature = TEMPERATURE,
    eval_map_every = 2,
    patience= PATIENCE,
)
end = time.perf_counter()
mlflow.log_metric('train_time_seconds', end-start)

In [ ]:
# model = SiameseBERT(model_name=MODEL_NAME)
# model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
# model.to(DEVICE)
# model.eval()
# model

In [ ]:

fragment_store = build_embedding_store(
    model, val_corpus, source_corpus, tokenizer, device=DEVICE
)
centroid_store = build_weighted_centroids(fragment_store)

frag_metrics, cent_metrics = compare_fragment_vs_centroid_metrics(
    model, val_corpus, fragment_store, centroid_store,
    tokenizer, k_values=K_VALUES, device=DEVICE
)

In [ ]:
# ── Пример предсказания ───────────────────────────────────────────────────────

sample_text = val_df['text'].iloc[777]
true_author = val_df['author'].iloc[777]

result = predict(
    text           = sample_text,
    model          = model,
    fragment_store = fragment_store,
    centroid_store = centroid_store,
    tokenizer      = tokenizer,
    device         = DEVICE
)

print(f"Текст (первые 100 символов): {sample_text[:100]}...")
print(f"Истинный автор: {true_author}")
print(f"Предсказанный:  {result['predicted']} (confidence={result['confidence']})")
print(f"Top-5:")
for author, score in result['top_k']:
    marker = '✓' if author == true_author else ' '
    print(f"  {marker} {author}: {score}")
print(result)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Опционально для UMAP: pip install umap-learn
# import umap

def get_embeddings_for_viz(store: EmbeddingStore,
                            max_per_author: int = 50):
    """
    Берём по max_per_author эмбеддингов от каждого автора.
    Ограничиваем чтобы график не был перегружен.
    """
    from collections import defaultdict

    author_indices = defaultdict(list)
    for i, author in enumerate(store.authors):
        author_indices[author].append(i)

    selected_indices = []
    selected_authors = []

    for author, indices in author_indices.items():
        chosen = indices[:max_per_author]
        selected_indices.extend(chosen)
        selected_authors.extend([author] * len(chosen))

    matrix = np.stack(store.embeddings)[selected_indices]
    return matrix, selected_authors


def reduce_2d(matrix: np.ndarray, method: str = 'tsne') -> np.ndarray:
    """
    Понижение размерности до 2D.

    Сначала PCA до 50 измерений — ускоряет t-SNE без потери структуры.
    Затем t-SNE или UMAP до 2D.

    method: 'tsne' или 'umap'
    """
    # PCA — предварительное сжатие
    n_pca = min(50, matrix.shape[1], matrix.shape[0] - 1)
    pca   = PCA(n_components=n_pca, random_state=RANDOM_STATE)
    reduced = pca.fit_transform(matrix)

    if method == 'tsne':
        tsne = TSNE(
            n_components=2,
            perplexity=min(30, len(matrix) // 5),
            random_state=RANDOM_STATE,
            n_iter=1000,
            metric='cosine'   # эмбеддинги нормализованы — косинус уместен
        )
        return tsne.fit_transform(reduced)

    elif method == 'umap':
        import umap
        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=15,
            min_dist=0.1,
            metric='cosine',
            random_state=RANDOM_STATE
        )
        return reducer.fit_transform(reduced)


def reduce_3d(matrix: np.ndarray) -> np.ndarray:
    """Понижение до 3D через UMAP — t-SNE в 3D работает хуже."""
    import umap
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.1,
        metric='cosine',
        random_state=RANDOM_STATE
    )
    n_pca   = min(50, matrix.shape[1], matrix.shape[0] - 1)
    reduced = PCA(n_components=n_pca, random_state=RANDOM_STATE).fit_transform(matrix)
    return reducer.fit_transform(reduced)


# ── 2D график ──────────────────────────────────────────────────────

def plot_2d(store: EmbeddingStore,
            method: str = 'tsne',
            max_per_author: int = 50,
            figsize: tuple = (14, 10)):

    matrix, authors = get_embeddings_for_viz(store, max_per_author)
    coords          = reduce_2d(matrix, method=method)
    unique_authors  = sorted(set(authors))

    # Цвета — tab20 для до 20 авторов, иначе hsv
    cmap   = plt.cm.get_cmap('tab20' if len(unique_authors) <= 20 else 'hsv',
                              len(unique_authors))
    colors = {a: cmap(i) for i, a in enumerate(unique_authors)}

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#0f0f1a')
    fig.patch.set_facecolor('#0f0f1a')

    for author in unique_authors:
        mask = [a == author for a in authors]
        xs   = coords[mask, 0]
        ys   = coords[mask, 1]
        ax.scatter(xs, ys,
                   c=[colors[author]],
                   label=author,
                   alpha=0.7,
                   s=25,
                   edgecolors='none')

    ax.set_title(f'Эмбеддинги авторов ({method.upper()})',
                 color='white', fontsize=14, pad=15)
    ax.tick_params(colors='gray')
    for spine in ax.spines.values():
        spine.set_color('#333355')

    legend = ax.legend(
        bbox_to_anchor=(1.02, 1), loc='upper left',
        fontsize=7, ncol=max(1, len(unique_authors) // 25),
        facecolor='#1a1a2e', edgecolor='#333355', labelcolor='white'
    )

    plt.tight_layout()
    plt.savefig(f'embeddings_2d_{method}.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()
    print(f"Сохранено: embeddings_2d_{method}.png")


# ── 3D интерактивный график через Plotly ──────────────────────────

def plot_3d(store: EmbeddingStore, max_per_author: int = 50):
    """
    Интерактивный 3D график — можно вращать мышью прямо в Colab.
    Требует: pip install plotly
    """
    import plotly.graph_objects as go

    matrix, authors = get_embeddings_for_viz(store, max_per_author)
    coords          = reduce_3d(matrix)
    unique_authors  = sorted(set(authors))

    traces = []
    for author in unique_authors:
        mask = [a == author for a in authors]
        traces.append(go.Scatter3d(
            x=coords[mask, 0],
            y=coords[mask, 1],
            z=coords[mask, 2],
            mode='markers',
            name=author,
            marker=dict(size=4, opacity=0.75)
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title='Эмбеддинги авторов (UMAP 3D)',
        paper_bgcolor='#0f0f1a',
        plot_bgcolor='#0f0f1a',
        font=dict(color='white'),
        scene=dict(
            xaxis=dict(backgroundcolor='#1a1a2e', gridcolor='#333355'),
            yaxis=dict(backgroundcolor='#1a1a2e', gridcolor='#333355'),
            zaxis=dict(backgroundcolor='#1a1a2e', gridcolor='#333355'),
        ),
        legend=dict(
            font=dict(size=9),
            itemsizing='constant'
        )
    )
    fig.show()  # интерактивно в Colab

In [ ]:
plot_2d(fragment_store, method='umap')

In [ ]:
mlflow.log_artifact('embeddings_2d_umap.png', 'plots/embeddings_2d_umap.png')

In [ ]:
plot_2d(centroid_store, method='umap')

In [ ]:
mlflow.log_artifact('embeddings_2d_umap.png', 'plots/embeddings_2d__centroids_umap.png')

In [ ]:
"""
Оценка модели по 1-NN предсказанию на отдельном тестовом датасете.

Логика:
  1. Reference store строится из TRAIN корпуса (известная база авторов).
  2. Каждый текст из TEST корпуса прогоняется через модель → эмбеддинг.
  3. Находим ближайший эмбеддинг в reference store → предсказанный автор.
  4. Сравниваем предсказанного автора с истинным → classification metrics.

Это принципиально отличается от MAP@K:
  - MAP@K оценивает качество ранжирования (есть ли правильный автор в top-K).
  - 1-NN accuracy оценивает точность предсказания: модель делает один выбор,
    и он либо правильный, либо нет.
"""

import numpy as np
import pandas as pd
import torch
from torch.cuda.amp import autocast
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

def evaluate_1nn(
    model,
    test_df: pd.DataFrame,
    reference_store,
    tokenizer,
    device: str     = "cuda",
    batch_size: int = 64,
    max_len: int    = 256,
    text_col: str   = "text",
    author_col: str = "author",
    print_report: bool = True,
) -> dict:
    """
    Для каждого текста в test_df находит ближайший эмбеддинг
    в reference_store и считает classification-метрики.
    """
    model.eval()

    test_texts   = test_df[text_col].tolist()
    true_authors = test_df[author_col].tolist()

    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(test_texts), batch_size):
            batch = test_texts[i : i + batch_size]
            enc = tokenizer(
                batch,
                max_length=max_len,
                padding=True,
                truncation=True,
                return_tensors="pt",
            )
            with autocast():
                emb = model.encode(
                    enc["input_ids"].to(device),
                    enc["attention_mask"].to(device),
                ).float()
            all_embeddings.append(emb.cpu().numpy())

    query_matrix = np.vstack(all_embeddings)

    store_matrix  = reference_store.get_matrix()
    store_authors = reference_store.get_authors()

    sim_matrix = query_matrix @ store_matrix.T

    nearest_idx    = np.argmax(sim_matrix, axis=1) # индекс в store
    pred_authors   = [store_authors[j] for j in nearest_idx]
    pred_sims      = sim_matrix[np.arange(len(nearest_idx)), nearest_idx]

    known_authors_in_store = set(store_authors)
    labels = sorted(set(true_authors) & known_authors_in_store)

    mask = [a in known_authors_in_store for a in true_authors]
    y_true = [a for a, m in zip(true_authors, mask) if m]
    y_pred = [a for a, m in zip(pred_authors, mask) if m]

    n_skipped = len(true_authors) - len(y_true)
    if n_skipped > 0:
        print(f"{n_skipped} тестовых текстов пропущено: "
              f"их авторы отсутствуют в reference store")

    results = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, labels=labels,
                                              average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, labels=labels,
                                           average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, labels=labels,
                                       average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, labels=labels,
                                              average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, labels=labels,
                                           average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, labels=labels,
                                       average="weighted", zero_division=0),
        "n_test": len(y_true),
        "n_authors": len(labels),
        "mean_confidence": float(pred_sims[mask].mean()) if any(mask) else 0.0,
    }

    if print_report:
        print(f"\n{'═' * 60}")
        print(f"  1-NN Classification Report  ({len(y_true)} текстов, "
              f"{len(labels)} авторов)")
        print(f"{'═' * 60}")
        print(f"  Accuracy:           {results['accuracy']:.4f}")
        print(f"  Precision (macro):  {results['precision_macro']:.4f}")
        print(f"  Recall    (macro):  {results['recall_macro']:.4f}")
        print(f"  F1        (macro):  {results['f1_macro']:.4f}")
        print(f"  F1      (weighted): {results['f1_weighted']:.4f}")
        print(f"  Mean cosine sim:    {results['mean_confidence']:.4f}")
        print(f"{'─' * 60}")
        print(classification_report(
            y_true, y_pred, labels=labels, zero_division=0
        ))

    results["y_true"] = y_true
    results["y_pred"] = y_pred
    results["labels"] = labels

    return results, classification_report(y_true, y_pred, labels=labels, zero_division=0)

In [ ]:
TRAIN_TEST_SPLIT = 0.2
mlflow.log_param('class_report_train_test_split', TRAIN_TEST_SPLIT)

In [ ]:
#local_path = client.download_artifacts(run_id=MLFLOW_DF_ARTIFACT_RUN_ID, path=MLFLOW_DF_ARTIFACT_PATH)
local_path = "balanced_lit.csv"
data_full = pd.read_csv(local_path)

In [ ]:
train_test_df, test_test_df = train_test_split(data_full, test_size=TRAIN_TEST_SPLIT, random_state=RANDOM_STATE)

In [ ]:
train_test_df.head()

In [ ]:
for author in EXCLUDED_AUTHORS_LIST:
  print(len(train_test_df[train_test_df.author.isin([author])]))
  print(len(test_test_df[test_test_df.author.isin([author])]))

In [ ]:
source_corpus = df_to_source_corpus_man(train_test_df, 'author')
train_corpus = df_to_corpus(train_test_df)

In [ ]:
ref_store = build_embedding_store(
    model, train_corpus, source_corpus,
    tokenizer, device=DEVICE
)

In [ ]:
metrics, cr_text = evaluate_1nn(
    model          = model,
    test_df        = test_test_df,
    reference_store = ref_store,
    tokenizer      = tokenizer,
    device         = DEVICE,
)

In [ ]:
mlflow.log_metric('accuracy_classification_report_02', metrics['accuracy'])
mlflow.log_metric('recall_macro_classification_report_02', metrics['recall_macro'])
mlflow.log_metric('precision_macro_classification_report_02', metrics['precision_macro'])

In [ ]:
with open('cr_02.txt', 'w') as f:
  f.write(cr_text)
  mlflow.log_artifact('cr_02.txt', 'data/cr.txt')

In [ ]:
import json
with open('cr_02_metrics.json', 'w') as f:
  json.dump(metrics, f, indent=4)
  mlflow.log_artifact('cr_02_metrics.json', 'data/metrics.json')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = metrics['y_true']
y_pred = metrics['y_pred']
labels = metrics['labels']

cm  = confusion_matrix(y_true, y_pred, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)  # row-normalised

n = len(labels)
figsize = max(14, n * 0.45)
fig, ax = plt.subplots(figsize=(figsize, figsize))

sns.heatmap(
    cm_norm,
    ax=ax,
    xticklabels=labels,
    yticklabels=labels,
    cmap='Blues',
    vmin=0, vmax=1,
    linewidths=0.3,
    linecolor='#e0e0e0',
    annot=(n <= 20),
    fmt='.2f' if n <= 20 else '',
    annot_kws={'size': 7},
    cbar_kws={'label': 'Доля предсказаний (по строке)'},
)

ax.set_xlabel('Предсказанный автор', fontsize=12, labelpad=10)
ax.set_ylabel('Истинный автор',  fontsize=12, labelpad=10)
ax.set_title(
    f'Матрица ошибок (1-NN, нормализована по строке)\n'
    f'Accuracy={metrics["accuracy"]:.3f}  F1-macro={metrics["f1_macro"]:.3f}  '
    f'n={metrics["n_test"]} текстов, {n} авторов',
    fontsize=13, pad=14
)
ax.tick_params(axis='x', rotation=90, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
mlflow.log_figure(fig, 'plots/confusion_matrix.png')
plt.show()

In [ ]:
AUTHORS_TO_INSPECT = EXCLUDED_AUTHORS_LIST

y_true = metrics['y_true']
y_pred = metrics['y_pred']

inspect_data = {'authors': []}

for author in AUTHORS_TO_INSPECT:
    indices = [i for i, a in enumerate(y_true) if a == author]
    if not indices:
        print(f'[{author}] — не найден в тестовой выборке\n')
        continue

    preds = [y_pred[i] for i in indices]
    total = len(preds)
    counts = {}
    for p in preds:
        counts[p] = counts.get(p, 0) + 1
    counts = sorted(counts.items(), key=lambda x: -x[1])

    predictions = [
        {'predicted_label': label, 'percentage': round(cnt / total * 100, 2)}
        for label, cnt in counts
    ]
    inspect_data['authors'].append({'true_author': author, 'predictions': predictions})

    correct = sum(c for p, c in counts if p == author)
    print(f'┌─ {author} ({total} фрагментов, accuracy {correct/total:.1%})')
    for pred_label, cnt in counts:
        pct  = cnt / total
        bar  = '█' * int(pct * 30)
        mark = ' ✓' if pred_label == author else ''
        print(f'│  {pred_label:<28} {bar:<30} {pct:>6.1%}  ({cnt}){mark}')
    print('└' + '─' * 70 + '\n')

with open('author_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(inspect_data, f, ensure_ascii=False, indent=2)
mlflow.log_artifact('author_predictions.json', 'data/author_predictions.json')
print('Сохранено в MLflow: data/author_predictions.json')

In [ ]:

torch.save(model.state_dict(), '/content/drive/MyDrive/best_model_ruslit_40_090626_1.pt')

In [ ]:
import mlflow.pytorch

mlflow.pytorch.save_model(pytorch_model=model, path="models/best_model_ruslit_40")

In [ ]:
mlflow.end_run()

In [ ]:
print('here')